 Definition: Medallian architecture is a data design pattern used to logically organize data store in lakehouse with the goal of incrementally and progressively imporving the quality of data as it flows through each layer. 

## Staging Directory

Read the customers data, products data and orders data. 

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
spark.sql(f'USE CATALOG {catalog_name};')

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS BRONZE;
CREATE SCHEMA IF NOT EXISTS SILVER;
CREATE SCHEMA IF NOT EXISTS GOLD; 

In [0]:
customer_df = spark.read.csv(
    "/Volumes/wns24082026/quickstart_schema/sandbox/datasets/e-commerce/staging/customers/customers_001.csv",
    header=True,
    inferSchema=True,
)

products_df = spark.read.json(
    path="/Volumes/wns24082026/quickstart_schema/sandbox/datasets/e-commerce/staging/products/"
)

orders_df = spark.read.json(
    path="/Volumes/wns24082026/quickstart_schema/sandbox/datasets/e-commerce/staging/orders/"
)

## Bronze Layer

In [0]:
customer_df.write.saveAsTable("bronze.customers",mode='OVERWRITE')
products_df.write.saveAsTable("bronze.products",mode = 'OVERWRITE')
orders_df.write.saveAsTable("bronze.orders",mode = 'OVERWRITE')

## Silver layer

Create a new column with total_price being price * quantity

In [0]:
from pyspark.sql.functions import col

customer_df = (
    spark.read.table("bronze.orders")
    .filter(col("order_id").isNotNull())
    .withColumn("total_price", col("qty") * col("price"))
).write.saveAsTable("silver.orders", mode="OVERWRITE")

## Gold Layer

In [0]:
from pyspark.sql.functions import sum

spark.read.table("silver.orders").groupBy("item_id").agg(
    sum("total_price").alias("total_price_sum")
).write.saveAsTable("gold.revenue_by_product", mode="OVERWRITE")